# 🕵️‍♂️ Dynamic Council Evaluation Explorer
Run the cell below to load your evaluation results and interactively read through the multi-persona reviews!

In [4]:
import json
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

results = []
try:
    with open('results_council_gemma4:26b.jsonl', 'r') as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
except FileNotFoundError:
    print("Run the dynamic_council.py script first to generate results.")
    
if results:
    correct = sum(1 for r in results if r.get('correct'))
    total = len(results)
    display(HTML(f"""
    <div style='background-color: #e8f4f8; color: #111; padding: 15px; border-radius: 8px; font-size: 18px; margin-bottom: 20px'>
        <b>Total Papers Evaluated:</b> {total} <br>
        <b>Council Accuracy:</b> {correct}/{total} ({correct/total*100:.1f}%) <br>
        <i>Based on {len(results[0].get('personas', []))} distinct persona profiles per paper</i>
    </div>
    """))

out = widgets.Output()

def view_paper(change):
    idx = change['new'] if change else 0
    if not results: return
    res = results[idx]
    gt = res['decision'].upper()
    pred = res.get('prediction', 'Unknown')
    pred = pred.upper() if pred else 'None'
    color = 'green' if res.get('correct') else '#d63031'
    
    with out:
        out.clear_output(wait=True)
        display(HTML(f"""
        <h2>📄 {res['title']}</h2>
        <h4 style='color: #666; margin-top: -10px'>OpenReview ID: <a href="{res['forum_url']}" target="_blank">{res['paper_id']}</a></h4>
        <div style='background-color: #f8f9fa; color: #111; padding: 15px; border-radius: 8px; border-left: 5px solid {color}'>
            <b>Ground Truth:</b> {gt} &nbsp;|&nbsp; 
            <b>Council Verdict:</b> <span style='color: {color}; font-weight: bold'>{pred}</span>
        </div>
        """))
        
        display(HTML("<hr><h3 style='color: #8ab4f8'>⚖️ Area Chair Consolidation Report</h3>"))
        display(Markdown(res.get('consolidation', 'No consolidation text logged.')))
        
        display(HTML("<hr><h3 style='color: #8ab4f8'>🕵️‍♂️ The Committee Reviews</h3>"))
        for rev in res.get('reviews', []):
            display(HTML("<div style='border-left: 4px solid #3498db; padding-left: 20px; padding-top: 5px; margin-bottom: 25px; background: rgba(52, 152, 219, 0.05)'>"))
            display(Markdown(rev))
            display(HTML("</div>"))

if results:
    options = [(f"[{'✓' if r.get('correct') else '✗'}] {r['title'][:60]}...", i) for i, r in enumerate(results)]
    dropdown = widgets.Dropdown(options=options, description='Select Paper:', layout={'width': '80%'}, style={'description_width': 'initial'})
    dropdown.observe(view_paper, names='value')
    view_paper(None)  # Render initial choice
    display(dropdown, out)

Dropdown(description='Select Paper:', layout=Layout(width='80%'), options=(('[✓] Retrieval-Augmented Generatio…

Output()

## Code for visualizing debate

In [8]:
import json
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

DEBATE_FILE = 'sample_debate.jsonl'  # ← change if needed

# ── Load results ─────────────────────────────────────────────────────────────
debate_results = []
try:
    with open(DEBATE_FILE) as f:
        for line in f:
            if line.strip():
                debate_results.append(json.loads(line))
    print(f'Loaded {len(debate_results)} paper(s) from {DEBATE_FILE}')
except FileNotFoundError:
    print(f'File not found: {DEBATE_FILE} — run debate_council.py first.')

# ── Shared CSS (injected once) ────────────────────────────────────────────────
display(HTML("""
<style>
.db-card   { background:#1e1e2e; color:#cdd6f4; padding:16px; border-radius:10px; margin-bottom:14px; }
.db-pro    { border-left:5px solid #89b4fa; }
.db-crit   { border-left:5px solid #f38ba8; }
.db-chair  { border-left:5px solid #a6e3a1; background:#1a2a1a; }
.db-badge  { display:inline-block; padding:2px 10px; border-radius:20px; font-size:12px; font-weight:bold; margin-right:6px; }
.db-valid  { background:#a6e3a1; color:#1e1e2e; }
.db-weak   { background:#f9e2af; color:#1e1e2e; }
.db-inval  { background:#f38ba8; color:#1e1e2e; }
.db-change { background:#89dceb; color:#1e1e2e; }
.db-label  { font-size:11px; text-transform:uppercase; letter-spacing:1px; opacity:0.6; margin-bottom:4px; }
.db-round  { background:#FFFFFF; border-radius:8px; padding:12px; margin-bottom:10px; }
pre.db-pre { white-space:pre-wrap; word-wrap:break-word; font-family:inherit; margin:0; font-size:14px; line-height:1.4; }
</style>
"""))


def verdict_badge(text):
    """Coloured badge based on [VALID]/[WEAK]/[INVALID] tag."""
    tu = str(text).upper()
    if '[VALID]'   in tu: return '<span class="db-badge db-valid">VALID</span>'
    if '[INVALID]' in tu: return '<span class="db-badge db-inval">INVALID</span>'
    if '[WEAK]'    in tu: return '<span class="db-badge db-weak">WEAK</span>'
    return ''


def render_analysis_table(items, role_color):
    """Render a list of claim-dicts as styled HTML blocks."""
    if not items:
        return '<p><em>No data</em></p>'
    # Handle PARSE_ERROR wrapper — try to unpack the embedded JSON string
    if len(items) == 1 and items[0].get('claim') == 'PARSE_ERROR':
        raw = items[0].get('verdict', '')
        try:
            clean = raw.replace('```json', '').replace('```', '').strip()
            start = clean.index('['); end = clean.rindex(']') + 1
            items = json.loads(clean[start:end])
        except Exception:
            return f"<pre class='db-pre'>{raw}</pre>"
    rows = []
    for i, item in enumerate(items, 1):
        if not isinstance(item, dict): continue
        claim   = item.get('claim',   '—')
        evid    = item.get('evidence', item.get('proof', '—'))
        verdict = item.get('verdict',  item.get('evaluation', '—'))
        badge   = verdict_badge(str(verdict))
        known   = {'claim', 'evidence', 'proof', 'verdict', 'evaluation'}
        extras  = {k: v for k, v in item.items() if k not in known}
        extra_html = ''.join(
            f"<div style='margin-top:6px'><span class='db-label'>{k}</span> <span style='opacity:.9'>{v}</span></div>"
            for k, v in extras.items() if v
        )
        rows.append(f"""
        <div class='db-round' style='border-left:3px solid {role_color}'>
          <div style='font-weight:bold; margin-bottom:6px; font-size:15px'>#{i} — {claim}</div>
          <div class='db-label'>Evidence</div>
          <div style='margin-bottom:10px; opacity:.85'>{evid}</div>
          <div class='db-label'>Verdict {badge}</div>
          <pre class='db-pre'>{verdict}</pre>
          {extra_html}
        </div>
        """)
    return ''.join(rows)


def render_position_changes(changes):
    """Render position changes as highlighted revision cards."""
    if not changes:
        return "<em style='opacity:.5'>No position changes this round.</em>"
    html = []
    for c in changes:
        html.append(f"""
        <div style='background:#2a2a3e;padding:12px;border-radius:6px;margin-bottom:8px'>
          <span class='db-badge db-change'>REVISED</span>
          <strong style='font-size:14px'>{c.get('claim','?')[:120]}</strong><br/>
          <div style='margin-top:8px'>
            <span class='db-label'>was →</span> <span style='opacity:.8'>{c.get('original_verdict','—')}</span><br/>
            <span class='db-label'>now →</span> <span>{c.get('revised_verdict','—')}</span>
          </div>
          <div style='margin-top:6px'><span class='db-label'>reason</span> <em style='opacity:.9'>{c.get('reason','—')}</em></div>
        </div>
        """)
    return ''.join(html)


def view_debate(change):
    idx = change['new'] if change else 0
    if not debate_results:
        return
    res    = debate_results[idx]
    gt     = res.get('decision', res.get('label', '?')).upper()
    pred   = (res.get('prediction') or 'None').upper()
    color  = '#a6e3a1' if res.get('correct') else '#f38ba8'
    rounds = res.get('debate_transcript', [])
    pro_p  = res.get('proponent_persona', {})
    crit_p = res.get('critic_persona', {})

    html_parts = []

    # ── Paper header ─────────────────────────────────────────────────────
    html_parts.append(f"""
    <div class='db-card' style='border-left:5px solid {color}'>
      <h2 style='margin:0 0 6px'>{res.get('title','')}</h2>
      <div style='opacity:.6; font-size:13px'>
        <a href="{res.get('forum_url','')}" target="_blank" style='color:#89b4fa'>{res.get('paper_id','')}</a>
        &nbsp;|&nbsp; Domain: <strong>{res.get('domain','—')}</strong>
        &nbsp;|&nbsp; Rounds completed: <strong>{res.get('rounds_completed', len(rounds))}</strong>
      </div>
      <div style='margin-top:10px; font-size:16px'>
        Ground Truth: <strong>{gt}</strong> &nbsp;|&nbsp;
        Prediction: <strong style='color:{color}'>{pred}</strong>
      </div>
      <div style='margin-top:6px; font-size:13px; opacity:.6'>
        Proponent: {res.get('proponent_model','?')} &nbsp;|&nbsp;
        Critic: {res.get('critic_model','?')} &nbsp;|&nbsp;
        Chair: {res.get('chair_model','?')}
      </div>
    </div>
    """)

    # ── Adversarial Personas ──────────────────────────────────────────────
    html_parts.append("<hr><h3 style='color:#cba6f7; margin-bottom:15px'>🎭 Adversarial Personas</h3>")
    html_parts.append(f"""
    <div style='display:grid; grid-template-columns:1fr 1fr; gap:16px; margin-bottom:20px'>
      <div class='db-card db-pro' style='margin-bottom:0'>
        <div class='db-label'>⚗️ Proponent</div>
        <div style='font-size:17px; font-weight:bold; margin-bottom:8px'>{pro_p.get('name','—')}</div>
        <div style='opacity:.85; font-size:14px; line-height:1.4'>{pro_p.get('mandate','—')}</div>
      </div>
      <div class='db-card db-crit' style='margin-bottom:0'>
        <div class='db-label'>🔬 Critic</div>
        <div style='font-size:17px; font-weight:bold; margin-bottom:8px'>{crit_p.get('name','—')}</div>
        <div style='opacity:.85; font-size:14px; line-height:1.4'>{crit_p.get('mandate','—')}</div>
      </div>
    </div>
    """)

    # ── Phase 2: Initial analyses ─────────────────────────────────────────
    html_parts.append("<hr><h3 style='color:#89b4fa; margin-bottom:15px'>📋 Phase 2 — Initial Independent Analyses</h3>")
    html_parts.append(f"""
    <div style='display:grid; grid-template-columns:1fr 1fr; gap:16px'>
      <div>
        <div class='db-label' style='color:#89b4fa; margin-bottom:8px; font-size:13px'>⚗️ Proponent — {pro_p.get('name','')}</div>
        {render_analysis_table(res.get('proponent_analysis', []), '#89b4fa')}
      </div>
      <div>
        <div class='db-label' style='color:#f38ba8; margin-bottom:8px; font-size:13px'>🔬 Critic — {crit_p.get('name','')}</div>
        {render_analysis_table(res.get('critic_analysis', []), '#f38ba8')}
      </div>
    </div>
    """)

    # ── Phase 3: Debate rounds ────────────────────────────────────────────
    html_parts.append("<hr><h3 style='color:#fab387; margin-bottom:15px'>🥊 Phase 3 — Cross-Examination Rounds</h3>")
    if not rounds:
        html_parts.append("<p><em>No debate rounds recorded.</em></p>")
        
    for rnd_idx, rnd in enumerate(rounds, 1):
        pro_r  = rnd.get('proponent', {})
        crit_r = rnd.get('critic', {})
        pro_s  = pro_r.get('structured', {})
        crit_s = crit_r.get('structured', {})
        pro_ch  = pro_s.get('position_changes', [])
        crit_ch = crit_s.get('position_changes', [])
        pro_sum  = pro_s.get('round_summary', '—')
        crit_sum = crit_s.get('round_summary', '—')

        html_parts.append(f"""
        <h4 style='color:#f9e2af; background:#1e1e2e; padding:10px 16px; border-radius:6px; margin:24px 0 12px 0; font-size:15px'>
          Round {rnd_idx}
          &nbsp;—&nbsp; Proponent revised {len(pro_ch)} position(s)
          &nbsp;|&nbsp; Critic revised {len(crit_ch)} position(s)
        </h4>""")

        html_parts.append(f"""
        <div style='display:grid; grid-template-columns:1fr 1fr; gap:16px; margin-bottom:18px'>
          <div class='db-card db-pro'>
            <div class='db-label' style='color:#89b4fa; font-size:13px; margin-bottom:10px'>⚗️ Proponent Rebuttal</div>
            <pre class='db-pre'>{pro_r.get('rebuttal','—')}</pre>
            <hr style='opacity:.2; margin:16px 0'>
            <div class='db-label' style='margin-bottom:8px'>Position Changes</div>
            {render_position_changes(pro_ch)}
            <div class='db-label' style='margin-top:12px'>Round Summary</div>
            <em style='opacity:.9'>{pro_sum}</em>
          </div>
          <div class='db-card db-crit'>
            <div class='db-label' style='color:#f38ba8; font-size:13px; margin-bottom:10px'>🔬 Critic Rebuttal</div>
            <pre class='db-pre'>{crit_r.get('rebuttal','—')}</pre>
            <hr style='opacity:.2; margin:16px 0'>
            <div class='db-label' style='margin-bottom:8px'>Position Changes</div>
            {render_position_changes(crit_ch)}
            <div class='db-label' style='margin-top:12px'>Round Summary</div>
            <em style='opacity:.9'>{crit_sum}</em>
          </div>
        </div>
        """)

    # ── Phase 4: Area Chair verdict ───────────────────────────────────────
    html_parts.append("<hr><h3 style='color:#a6e3a1; margin-bottom:15px'>⚖️ Phase 4 — Area Chair Verdict</h3>")
    verdict_text = res.get('verdict', 'No verdict recorded.')
    html_parts.append(f"<div class='db-card db-chair' style='padding:20px'><pre class='db-pre'>{verdict_text}</pre></div>")

    final_html = "".join(html_parts)

    with debate_out:
        debate_out.clear_output(wait=True)
        display(HTML(final_html))


if debate_results:
    correct_d  = sum(1 for r in debate_results if r.get('correct'))
    total_d    = len(debate_results)
    avg_rounds = sum(r.get('rounds_completed', 0) for r in debate_results) / total_d
    display(HTML(f"""
    <div style='background:#1e1e2e; color:#cdd6f4; padding:16px; border-radius:10px;
                border-left:5px solid #cba6f7; font-size:16px; margin-bottom:20px'>
      <b>Debate Council Summary</b><br/>
      Papers evaluated: <b>{total_d}</b> &nbsp;|&nbsp;
      Correct decisions: <b>{correct_d}/{total_d} ({correct_d/total_d*100:.1f}%)</b> &nbsp;|&nbsp;
      Avg rounds: <b>{avg_rounds:.1f}</b>
    </div>
    """))

    debate_out = widgets.Output()
    options = [
        (f"[{'✓' if r.get('correct') else '✗'}] {r.get('title', '')[:65]}", i)
        for i, r in enumerate(debate_results)
    ]
    dd = widgets.Dropdown(
        options=options,
        description='Paper:',
        layout={'width': '90%'},
        style={'description_width': 'initial'}
    )
    dd.observe(view_debate, names='value')
    view_debate(None)
    display(dd, debate_out)


Loaded 1 paper(s) from sample_debate.jsonl


Dropdown(description='Paper:', layout=Layout(width='90%'), options=(('[✓] Geometry-aware Instance-reweighted A…

Output()